# 🚀 Fine-Tune CareerGPT Small LLM (Qwen2.5-0.5B-Instruct)

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using **Unsloth** and **QLoRA** for structured resume parsing, executive summarization, and interview feedback.

### 📌 Instructions for Google Colab:
1. Open this notebook in **Google Colab**.
2. Enable GPU: Go to `Runtime` -> `Change runtime type` -> Select `T4 GPU`.
3. Run all cells sequentially (`Ctrl + F9`).
4. Download the generated `Qwen2.5-0.5B-Instruct.Q4_K_M.gguf` file.
5. Rename and paste the downloaded `.gguf` file into your local project directory at:
   `models/fine_tuned_weights/careergpt_0.5b.gguf`

## Step 1: Install Unsloth, Hugging Face & Dependencies

In [ ]:
# Install Unsloth and unsloth_zoo for Colab Python 3.12 compatibility
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers" "peft" "accelerate" "bitsandbytes" "trl" "datasets"

## Step 2: Load 4-bit Quantized Base Model (Qwen2.5-0.5B)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Add LoRA Adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✓ Model and LoRA adapters loaded successfully!")

## Step 3: Prepare Training Dataset for CareerGPT Summarization & Parsing

In [ ]:
from datasets import Dataset

# Sample dataset for resume summarization, ATS bullet optimization, and interview feedback
training_samples = [
    {
        "instruction": "Synthesize an executive professional summary from raw candidate experience.",
        "input": "Candidate with 2 years in Python, FastAPI, Docker, and PostgreSQL. Built microservices.",
        "output": "Results-driven Software Engineer with 2 years of experience building scalable microservices using Python, FastAPI, Docker, and PostgreSQL."
    },
    {
        "instruction": "Transform weak bullet point into quantifiable achievement bullet.",
        "input": "Worked on machine learning model for customer sentiment.",
        "output": "Engineered and deployed an NLP sentiment classification model using PyTorch, improving prediction accuracy by [X%] and reducing inference latency by [Y ms]."
    },
    {
        "instruction": "Generate 30-day skill gap action plan summary.",
        "input": "Target Role: MLOps Engineer. Missing Skills: Docker, Kubernetes, MLflow.",
        "output": "30-Day Strategy for MLOps Engineer: Focus Days 1–10 on Docker containerization, Days 11–20 on Kubernetes orchestration, and Days 21–30 on deploying a full MLflow tracking pipeline."
    }
]

# Format into Chat Template
formatted_data = []
for sample in training_samples:
    text = f"<|im_start|>system\n{sample['instruction']}<|im_end|>\n<|im_start|>user\n{sample['input']}<|im_end|>\n<|im_start|>assistant\n{sample['output']}<|im_end|>"
    formatted_data.append({"text": text})

dataset = Dataset.from_list(formatted_data)
print(f"✓ Dataset formatted with {len(dataset)} training samples.")

## Step 5: Fine-Tune & Export GGUF

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("🎉 Fine-tuning complete!")

# Export fine-tuned model directly into 4-bit GGUF format (~350 MB)
model.save_pretrained_gguf("careergpt_0.5b_weights", tokenizer, quantization_method = "q4_k_m")

# Download the GGUF weights to your local machine
from google.colab import files
files.download("careergpt_0.5b_weights_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf")